# Taller: Creación y Visualización de KPIs

## Objetivos de Aprendizaje

* **De Conocimientos:** Evidenciar cómo un Sistema de Información Gerencial (MIS) sintetiza los datos atómicos y transaccionales de un *Transaction Processing System* (TPS) en un repositorio común para generar reportes estructurados que apoyen la toma de decisiones medibles (KPIs).
* **De Destrezas:** Aplicar el enfoque de sistemas para resolver un problema de TI organizacional, mapeando el flujo desde los *Data Items* elementales hasta el *Knowledge* estratégico.
* **De Valores y Actitudes:** Experimentar el trabajo en equipo a través del codiseño de una solución de software empresarial y la presentación de argumentos técnicos efectivos.

## Introducción (Enlace Arquitectónico)

Recordemos que los **Sistemas de Procesamiento de Transacciones (TPS)** se encargan de monitorear, recolectar, almacenar y procesar los datos de las operaciones básicas diarias de la organización (datos en tiempo real, de alto volumen y nivel atómico). 

Por otro lado, los **Sistemas de Información Gerencial (MIS)** acceden a estos repositorios comunes de datos transaccionales para resumirlos, procesarlos y presentar reportes estructurados (como indicadores clave de rendimiento o **KPIs**) orientados a mandos medios y gerenciales. Nuestro objetivo de hoy es construir la capa del MIS que consume los datos del TPS que ustedes ya desarrollaron.

## 1: Construir Conocimiento desde los Datos

### Descripción del Sistema (TPS)

El TPS es un **sistema de agendamiento de citas médicas** que registra tres entidades transaccionales: médicos, pacientes y citas. Cada cita generada por el TPS representa una transacción atómica que vincula a un paciente con un médico en una fecha y motivo específicos.

---

### Matriz Conceptual de los 3 KPIs Seleccionados

#### KPI 1 — Volumen de Citas por Especialidad Médica

| Nivel | Descripción |
|---|---|
| **Data Items (Datos Crudos)** | `id_cita`, `id_medico` registrados  en `data_citas.txt` y el atributo `especialidad` del médico asignado en `data_medicos.txt`. Cada fila de citas es un hecho elemental sin contexto. |
| **Information (Información Organizada)** | Al hacer un *join* entre citas y médicos y agrupar por `especialidad`, se obtiene el conteo total de citas agendadas para cada área. Esto le da significado de demanda relativa a cada servicio. |
| **Knowledge (Conocimiento Aplicado)** | Si una especialidad supera consistentemente el promedio de citas, la gerencia sabe que debe asignar más horas médicas o contratar un especialista adicional. Si una especialidad está por debajo, puede evaluar reducir su disponibilidad horaria para optimizar costos. |

---

#### KPI 2 — Distribución Etaria de los Pacientes Atendidos

| Nivel | Descripción |
|---|---|
| **Data Items (Datos Crudos)** | `cedula_paciente` en `data_citas.txt` y el atributo `edad` del paciente en `data_pacientes.txt`. Son valores numéricos crudos sin clasificación. |
| **Information (Información Organizada)** | Al cruzar las citas con la tabla de pacientes y segmentar la edad en grupos etarios estándar (Niños 0-12, Adolescentes 13-17, Adultos Jóvenes 18-35, Adultos 36-59, Adultos Mayores 60+), se obtiene la composición demográfica real de la demanda del centro. |
| **Knowledge (Conocimiento Aplicado)** | Un dominio mayoritario del segmento adulto mayor indica que el centro debe reforzar especialidades geriátricas (Cardiología, Neurología). Un pico en el grupo infantil señala la necesidad de fortalecer Pediatría. Este KPI guía las decisiones de oferta de servicios. |

---

#### KPI 3 — Tasa de Atención por Médico (Citas Completadas vs. Pendientes)

| Nivel | Descripción |
|---|---|
| **Data Items (Datos Crudos)** | `id_medico` y el campo `estado` de cada cita en `data_citas.txt`, donde `0 = Pendiente` y `1 = Atendida`. Son marcas binarias individuales sin contexto de desempeño. |
| **Information (Información Organizada)** | Al agrupar por médico y calcular la razón entre citas atendidas y citas totales, se construye la **Tasa de Atención** (%) de cada profesional, permitiendo comparar el nivel de cumplimiento entre doctores. |
| **Knowledge (Conocimiento Aplicado)** | Una tasa de atención del 0% en un médico con múltiples citas puede indicar ausencias no reportadas, sobrecarga de agenda o bloqueos en el sistema. La gerencia puede usar este KPI como umbral de alerta para gestión de RRHH o revisión de agendas en tiempo real. |

> **Nota sobre calidad de datos (GIGO):** Se verificó que todos los `id_medico` en `data_citas.txt` existen en `data_medicos.txt` y que todos los `cedula_paciente` existen en `data_pacientes.txt`. No se encontraron valores nulos ni registros huérfanos, por lo que los KPIs calculados tienen integridad referencial completa.

## 2: Mapeo de Atributos y Boceto de Interfaz

### 2.1 Mapeo en el Esquema de Base de Datos

El sistema cuenta con tres tablas (archivos `.txt` delimitados por `|`):

#### Entidades y Atributos

| Tabla | Atributos |
|---|---|
| `data_medicos` | `id_medico` (PK), `nombre`, `edad`, `especialidad`, `horario`, `password` |
| `data_pacientes` | `cedula` (PK), `nombre`, `edad`, `telefono`, `password` |
| `data_citas` | `id_cita` (PK), `cedula_paciente` (FK → pacientes), `id_medico` (FK → medicos), `fecha`, `motivo`, `estado` |

#### Relaciones utilizadas por KPI

```
data_citas.id_medico ──── data_medicos.id_medico   → KPI 1 y KPI 3
data_citas.cedula_paciente ── data_pacientes.cedula → KPI 2
```

#### Atributos críticos por KPI

| KPI | Tabla principal | Atributos usados |
|---|---|---|
| KPI 1 – Citas por Especialidad | `data_citas` + `data_medicos` | `id_medico`, `especialidad` |
| KPI 2 – Distribución Etaria | `data_citas` + `data_pacientes` | `cedula_paciente`, `edad` |
| KPI 3 – Tasa de Atención | `data_citas` + `data_medicos` | `id_medico`, `nombre`, `estado` |

---

### 2.2 Bocetado Visual (Mockup)

El dashboard se organiza en **una fila de tres paneles** dentro de una sola figura `matplotlib`:

```
┌─────────────────────────────────────────────────────────────────────────┐
│            DASHBOARD KPIs — CLÍNICA MÉDICA                              │
├────────────────────┬────────────────────┬────────────────────────────────┤
│  KPI 1             │  KPI 2             │  KPI 3                         │
│  Barras            │  Barras            │  Barras agrupadas              │
│  horizontales      │  verticales        │  (Pendientes | Atendidas)      │
│  por especialidad  │  por grupo etario  │  por médico                    │
│                    │                    │                                │
│  Justificación:    │  Justificación:    │  Justificación:                │
│  Comparativa de    │  Distribución      │  Comparativa de estado de      │
│  categorías →      │  demográfica →     │  cumplimiento por persona →    │
│  barras horiz.     │  barras vert.      │  barras agrupadas              │
└────────────────────┴────────────────────┴────────────────────────────────┘
```

## 3: Construcción e Implementación de los KPIs

### Instrucciones Técnicas:
1. **Conexión al Repositorio:** El MIS que estamos construyendo no debe tener el mismo repositorio Git que el TPS. El MIS debe poder tener acceso a la base de datos o dataset de transacciones de su TPS.
2. **Procesamiento de Datos:** Utilicen las librerías de análisis de datos (como `pandas`) para cargar los datos crudos, realizar las uniones (*joins*), agrupaciones (*groupby*) y cálculos requeridos por las reglas de negocio de sus 3 KPIs.
3. **Generación de las Visualizaciones:** Programen la capa gráfica de su MIS (utilizando librerías como `matplotlib`, o `seaborn`). Asegúrense de que los gráficos sean limpios, estén correctamente etiquetados (ejes, títulos, leyendas) y correspondan fielmente a los mockups diseñados en el Paso 2.

In [ ]:

# PASO 3.1 — IMPORTACIONES Y CARGA DE DATOS

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Cargar los tres archivos del TPS
medicos = pd.read_csv(
    'data_medicos.txt', sep='|', header=None,
    names=['id_medico', 'nombre', 'edad', 'especialidad', 'horario', 'pass']
)

pacientes = pd.read_csv(
    'data_pacientes.txt', sep='|', header=None,
    names=['cedula', 'nombre', 'edad', 'telefono', 'pass']
)

citas = pd.read_csv(
    'data_citas.txt', sep='|', header=None,
    names=['id_cita', 'cedula_paciente', 'id_medico', 'fecha', 'motivo', 'estado']
)

# Convertir fecha a tipo datetime para posibles análisis temporales
citas['fecha'] = pd.to_datetime(citas['fecha'], format='%d-%m-%Y')

print(f" Médicos cargados   : {len(medicos)} registros")
print(f" Pacientes cargados : {len(pacientes)} registros")
print(f" Citas cargadas     : {len(citas)} registros")

In [ ]:

# PASO 3.2 — CÁLCULO DE LOS 3 KPIs


# KPI 1: Volumen de citas por especialidad 
df_kpi1 = citas.merge(medicos[['id_medico', 'especialidad']], on='id_medico')
kpi1 = (
    df_kpi1.groupby('especialidad')
    .size()
    .sort_values(ascending=True)   
    .reset_index(name='num_citas')
)

#KPI 2: Distribución etaria de pacientes con citas 
df_kpi2 = citas.merge(pacientes[['cedula', 'edad']], left_on='cedula_paciente', right_on='cedula')
bins   = [0, 12, 17, 35, 59, 150]
labels = ['Niños\n(0–12)', 'Adolescentes\n(13–17)',
          'Adultos Jóvenes\n(18–35)', 'Adultos\n(36–59)', 'Adultos Mayores\n(60+)']
df_kpi2['grupo_etario'] = pd.cut(df_kpi2['edad'], bins=bins, labels=labels, right=True)
kpi2 = df_kpi2['grupo_etario'].value_counts().reindex(labels).reset_index()
kpi2.columns = ['grupo', 'conteo']
kpi2['conteo'] = kpi2['conteo'].fillna(0).astype(int)

# KPI 3: Tasa de atención (pendiente vs. atendida) por médico
df_kpi3 = citas.merge(medicos[['id_medico', 'nombre']], on='id_medico')
kpi3 = (
    df_kpi3.groupby(['nombre', 'estado'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
kpi3.columns.name = None
if 0 not in kpi3.columns: kpi3[0] = 0
if 1 not in kpi3.columns: kpi3[1] = 0
kpi3.rename(columns={0: 'pendientes', 1: 'atendidas'}, inplace=True)
kpi3['total']         = kpi3['pendientes'] + kpi3['atendidas']
kpi3['tasa_atencion'] = (kpi3['atendidas'] / kpi3['total'] * 100).round(1)
kpi3['nombre_corto']  = kpi3['nombre'].str.replace(r'Dr\. |Dra\. ', '', regex=True)
kpi3 = kpi3.sort_values('total', ascending=False)

print("KPI 1 — Citas por Especialidad:")
print(kpi1.to_string(index=False))

print("\nKPI 2 — Distribución Etaria:")
print(kpi2.to_string(index=False))

print("\nKPI 3 — Tasa de Atención por Médico:")
print(kpi3[['nombre_corto','pendientes','atendidas','total','tasa_atencion']].to_string(index=False))

In [ ]:

# PASO 3.3 — DASHBOARD: VISUALIZACIÓN DE LOS 3 KPIs


# Paleta de colores institucional
COLOR_PRIMARY   = '#2563EB'   # azul oscuro
COLOR_SECONDARY = '#10B981'   # verde
COLOR_ACCENT    = '#F59E0B'   # ámbar
COLOR_DANGER    = '#EF4444'   # rojo
COLOR_BG        = '#F8FAFC'
COLOR_GRID      = '#E2E8F0'

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.patch.set_facecolor(COLOR_BG)
fig.suptitle(
    'Dashboard de KPIs — Sistema de Agendamiento de Citas Médicas',
    fontsize=15, fontweight='bold', color='#1E293B', y=1.01
)

# ── PANEL 1: Barras horizontales — Citas por Especialidad 
ax1 = axes[0]
ax1.set_facecolor(COLOR_BG)
bars1 = ax1.barh(
    kpi1['especialidad'], kpi1['num_citas'],
    color=COLOR_PRIMARY, edgecolor='white', height=0.6
)
# Etiquetas de valor al extremo de cada barra
for bar in bars1:
    width = bar.get_width()
    ax1.text(
        width + 0.05, bar.get_y() + bar.get_height() / 2,
        f'{int(width)}', va='center', ha='left',
        fontsize=11, fontweight='bold', color='#1E293B'
    )
ax1.set_xlabel('Número de Citas', fontsize=10, color='#475569')
ax1.set_title('KPI 1\nVolumen de Citas por Especialidad',
              fontsize=11, fontweight='bold', color='#1E293B', pad=12)
ax1.set_xlim(0, kpi1['num_citas'].max() + 0.8)
ax1.spines[['top', 'right', 'left']].set_visible(False)
ax1.tick_params(axis='y', labelsize=9, colors='#475569')
ax1.tick_params(axis='x', labelsize=9, colors='#475569')
ax1.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax1.grid(axis='x', color=COLOR_GRID, linewidth=0.8, linestyle='--')
# Destacar la especialidad líder
max_idx = kpi1['num_citas'].idxmax()
bars1[max_idx].set_color(COLOR_SECONDARY)

# ── PANEL 2: Barras verticales — Distribución Etaria 
ax2 = axes[1]
ax2.set_facecolor(COLOR_BG)
palette2 = [COLOR_SECONDARY if v == kpi2['conteo'].max() else COLOR_PRIMARY
            for v in kpi2['conteo']]
bars2 = ax2.bar(
    kpi2['grupo'], kpi2['conteo'],
    color=palette2, edgecolor='white', width=0.6
)
for bar in bars2:
    h = bar.get_height()
    if h > 0:
        ax2.text(
            bar.get_x() + bar.get_width() / 2, h + 0.1,
            f'{int(h)}', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color='#1E293B'
        )
ax2.set_ylabel('Número de Pacientes', fontsize=10, color='#475569')
ax2.set_title('KPI 2\nDistribución Etaria de Pacientes con Citas',
              fontsize=11, fontweight='bold', color='#1E293B', pad=12)
ax2.set_ylim(0, kpi2['conteo'].max() + 1.5)
ax2.spines[['top', 'right', 'left']].set_visible(False)
ax2.tick_params(axis='x', labelsize=8.5, colors='#475569')
ax2.tick_params(axis='y', labelsize=9,   colors='#475569')
ax2.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax2.grid(axis='y', color=COLOR_GRID, linewidth=0.8, linestyle='--')

# ── PANEL 3: Barras agrupadas — Tasa de Atención por Médico
ax3 = axes[2]
ax3.set_facecolor(COLOR_BG)
x      = np.arange(len(kpi3))
width3 = 0.35
bars_p = ax3.bar(x - width3/2, kpi3['pendientes'], width3,
                 label='Pendientes', color=COLOR_DANGER,   edgecolor='white')
bars_a = ax3.bar(x + width3/2, kpi3['atendidas'],  width3,
                 label='Atendidas',  color=COLOR_SECONDARY, edgecolor='white')
for bar in list(bars_p) + list(bars_a):
    h = bar.get_height()
    if h > 0:
        ax3.text(
            bar.get_x() + bar.get_width() / 2, h + 0.05,
            f'{int(h)}', ha='center', va='bottom',
            fontsize=9, fontweight='bold', color='#1E293B'
        )
ax3.set_xticks(x)
ax3.set_xticklabels(kpi3['nombre_corto'], rotation=35, ha='right',
                    fontsize=8.5, color='#475569')
ax3.set_ylabel('Número de Citas', fontsize=10, color='#475569')
ax3.set_title('KPI 3\nCitas Pendientes vs. Atendidas por Médico',
              fontsize=11, fontweight='bold', color='#1E293B', pad=12)
ax3.set_ylim(0, kpi3[['pendientes','atendidas']].max().max() + 1.5)
ax3.spines[['top', 'right', 'left']].set_visible(False)
ax3.tick_params(axis='y', labelsize=9, colors='#475569')
ax3.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax3.grid(axis='y', color=COLOR_GRID, linewidth=0.8, linestyle='--')
ax3.legend(fontsize=9, framealpha=0.9, loc='upper right')

plt.tight_layout(pad=2.5)
plt.savefig('dashboard_kpis.png', dpi=150, bbox_inches='tight',
            facecolor=COLOR_BG)
plt.show()
print(" Dashboard guardado como 'dashboard_kpis.png'")

## 4: Validación de Atributos de Información e Interpretación Gerencial

### 4.1 Matriz de Atributos de Valor

| Atributo | KPI 1 — Citas por Especialidad | KPI 2 — Distribución Etaria | KPI 3 — Tasa de Atención |
|---|---|---|---|
| **Simplicidad** |  Barras horizontales con una variable . Lectura directa sin sobrecargar al observador. |  Cinco grupos etarios familiares con colores diferenciados. El segmento más alto se destaca visualmente. |  Dos barras por médico con colores semánticos (rojo = pendiente, verde = atendida) permiten una comparativa inmediata. |
| **Oportunidad (Timeliness)** |  El cálculo es un simple `groupby` sobre datos planos. Se ejecuta en milisegundos, apto para reportes diarios. | El `pd.cut` y `value_counts` son operaciones O(n). Escala sin problema a miles de pacientes. |  La agregación por médico es instantánea. Permite alertas en tiempo real si se integra a un TPS en vivo. |
| **Accesibilidad** |  Un administrador no técnico identifica en segundos cuál especialidad tiene mayor demanda leyendo el eje. |  Los grupos etarios usan lenguaje natural ("Niños", "Adultos Mayores"), sin jerga técnica. | 

---

### 4.2 Documentación de Interpretación Gerencial

#### KPI 1 — Volumen de Citas por Especialidad
> *"Si este indicador muestra que Traumatología concentra el mayor volumen de citas (3 citas vs. 2 del resto de especialidades), la acción correctiva o estratégica que la gerencia debe ejecutar de inmediato es evaluar la ampliación del horario del Dr. Andres Castro o la contratación de un segundo traumatólogo. Adicionalmente, deberá revisarse si la alta demanda en esta especialidad está asociada a una temporada específica (deportiva, escolar) para anticipar la planificación de recursos con suficiente antelación."*

#### KPI 2 — Distribución Etaria de Pacientes con Citas
> *"Si este indicador revela que el 41% de las citas corresponden al segmento de Adultos Jóvenes (18–35 años), seguido de un empate entre Adultos (36–59) y Adultos Mayores (60+) con un 24% cada uno, la acción estratégica que la gerencia debe ejecutar es fortalecer los servicios orientados a población joven-adulta (Dermatología, Medicina General) y mantener la capacidad en especialidades geriátricas (Cardiología, Neurología). Si el segmento de Adultos Mayores crece en periodos posteriores, la gerencia deberá priorizar la habilitación de turnos en horarios matutinos adaptados a esta población."*

#### KPI 3 — Tasa de Atención por Médico
> *"Si este indicador muestra que el 75% de los médicos registra una tasa de atención del 0% (todas sus citas siguen en estado Pendiente), mientras que únicamente el Dr. Luis Mendez y la Dra. Carmen Vega han completado el 50% de sus citas, la acción correctiva que la gerencia debe ejecutar de inmediato es verificar si las citas pendientes corresponden a fechas futuras aún no alcanzadas (lo que es operacionalmente normal) o si existen citas vencidas no atendidas sin justificación, en cuyo caso se debe activar el protocolo de seguimiento al paciente y notificación al médico responsable para garantizar la continuidad asistencial."*